In [4]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from z3 import *
from typing import List
import pickle 

In [5]:
def import_neural(filename):
    model= tf.keras.models.load_model(filename)
    return model
model= import_neural("./GCD_NN")

In [6]:
weights=model.get_weights()

In [7]:
def size_layers(model):
    size_layer=[]
    for i in range(0,len(model),2):
        size_layer.append(model[i].shape)
    return size_layer
size_layer= size_layers(weights)
size_layer

[(10, 10), (10, 1)]

In [8]:
def create_smt_file(name_file):
    file= open(name_file,'w')
    return file
def close_smt_file(f):
    f.close()

In [9]:
file= create_smt_file('model.smt2')

In [10]:
def initialize_variables(model):
    file.write("\n")
    
    size_layer= size_layers(model)
    
    for num_layer in range(len(size_layer)):
        
        for i in range(size_layer[num_layer][0]):
            file.write("(declare-fun i"+str(num_layer)+str(i)+"() Real)\n")
        file.write("\n") 
        
        for j in range(size_layer[num_layer][1]):
            file.write("(declare-fun y"+str(num_layer)+str(j)+"() Real)\n")
        file.write("\n") 
        
        for k in range(size_layer[num_layer][1]):
            file.write("(declare-fun o"+str(num_layer)+str(k)+"() Real)\n")
        file.write("\n") 
            
    file.write("(declare-fun decision() Int)\n")

initialize_variables(weights)

In [11]:
def constraint_input_output(model):
    
    size_layer= size_layers(model)
    file.write("\n") 
    for num_layer in range(len(size_layer)):
        if num_layer==0:
            file.write("\n") 
            for i in range(size_layer[num_layer][0]):
                file.write("(assert (or \n")
                file.write("(= i0"+str(i)+" 1) (= i0"+str(i)+" 0)\n")
                file.write("))\n")
                
        else:
            file.write("\n") 
            for j in range(size_layer[num_layer][0]):
                file.write("(assert \n")
                file.write("(= i"+str(num_layer)+str(j)+" o"+str(num_layer-1)+str(j)+")\n")
                file.write(")\n")
            
    
    file.write("(assert (or \n")
    file.write("(= decision 1) (= decision 0)\n")
    file.write("))\n")

constraint_input_output(weights)

In [12]:
def weight_computation(model):
    
    size_layer= size_layers(model)
    file.write("\n") 
    for num_layer in range(len(size_layer)):
        print(num_layer)
        for i in range(size_layer[num_layer][1]):
            file.write("(assert (= \n")
            file.write("y"+str(num_layer)+str(i)+" \n")
            file.write("(+\n")
            
            for j in range(size_layer[num_layer][0]):
                file.write("(* " +str(model[num_layer*2][j][i])+" i"+str(num_layer)+str(j)+")\n")
            file.write("(+ " +str(model[num_layer*2 + 1][i])+" 0)\n")
            #print((model[num_layer*2 + 1][i][0]))
            file.write(")))\n")

weight_computation(weights)

0
1


In [13]:
def compute_output(model):
    size_layer= size_layers(model)
    file.write("\n") 
    for num_layer in range(len(size_layer)):
        
        for i in range(size_layer[num_layer][1]):
            file.write("(assert (and \n")
            file.write("(or (not \n")
            file.write("(> y"+str(num_layer)+str(i)+ " 0))\n")
            file.write("(= o"+str(num_layer)+str(i)+ " y"+str(num_layer)+str(i)+"))\n")
            file.write("(or (not \n")
            file.write("(<= y"+str(num_layer)+str(i)+ " 0))\n")
            file.write("(= o"+str(num_layer)+str(i)+ " 0))\n")
            file.write("))\n")       
            
compute_output(weights)

In [14]:
def compute_decision(model, threshold):
    size_layer= size_layers(model)
    file.write("\n") 
    file.write("(assert (and \n")
    file.write("(or (not \n")
    file.write("(> o"+str(len(size_layer)-1)+"0 "+str(threshold)+"))\n")
    file.write("(= decision 1))\n")
    file.write("(or (not \n")
    file.write("(<= o"+str(len(size_layer)-1)+"0 "+str(threshold)+"))\n")
    file.write("(= decision 0))\n")
    file.write("))\n")
    
compute_decision(weights,0.5)

In [15]:
close_smt_file(file)

#### Create test profiles

This notebook allows to compute the decisions of the test set with the trained neural network.

In [18]:
def load_profiles(path):
    data= pd.read_csv(path, sep=',')
    data= data.drop(['Unnamed: 0','Risk'], axis=1)
    return(data)
profile= load_profiles('binarized_GCD')
profs= profile.values
profs[0:].shape

(1000, 10)

In [19]:
def compute_decisions(model,profiles,Threshold):
    individuals=[]
    predictions= model.predict(profiles[0:])
    y=predictions.tolist()
    for elem in y:
        if elem[0] < Threshold:
            individuals.append(0)
        else:
            individuals.append(1)
    return individuals
predictions=compute_decisions(model,profs,0.5)

In [20]:
def save_individuals(dataframe, predictions,new_column):
    dataframe[new_column] = predictions
    dataframe.to_csv("GCD_individuals", sep=',')
    return dataframe
df= save_individuals(profile,predictions,'y')

In [21]:
path= "model.smt2"
def load_solver(path):
    s= Solver()
    s.from_file(path)
    return s
solver= load_solver(path)

In [22]:
def check_sat(s, profil):
    predictions=[]
    for prof in profil:
        s.push()
        for i in range(len(prof)):
            s.add(Real('i0'+str(i)) ==  prof[i])
        s.add(Int('decision') == 0)
        resp= s.check()
        if resp== unsat:
            predictions.append(1)
        else:
            predictions.append(0)
        s.pop()
    return predictions
predictions_smt=check_sat(solver,profs)
save_individuals(df,predictions_smt,'y_smt')

,A,G,J,H,S,B,C,D,P,M,y,y_smt
0,1,1,1,1,0,0,0,0,0,1,1,1
1,0,0,1,1,1,0,1,1,0,0,1,1
2,1,1,0,1,1,1,0,0,0,1,1,1
3,1,1,1,0,1,0,1,1,0,1,0,0
4,1,1,1,0,1,0,1,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,1,1,1,0,0,0,0,1,1
996,1,1,0,1,1,0,1,1,1,0,1,1
997,1,1,1,1,1,1,0,0,0,1,1,1
998,0,1,1,0,1,0,0,1,0,1,0,0
